# Task 3 – Kakuro Constraint Satisfaction Problem (CSP) Experiments
**Owner:** Member 2 (`person-b`)  
**Course:** 503043 – Introduction to Artificial Intelligence

This notebook evaluates our Kakuro CSP formulation, Constraint Propagation (`Forward Checking` vs. `Generalized Arc Consistency`), and Heuristics (`MRV`, `MRV+Degree`, `LCV`) on both official boards from the assignment specification:
1. **`small_5x5.json`** (Page 4 example board: 12 white cells, 8 sum runs, total clue sum = 60).
2. **`large_9x9.json`** (Page 5 complex test case: 48 white cells, 28 sum runs, total clue sum = 249).

In [1]:
from pathlib import Path
import csv
import statistics
from task3_kakuro.model import KakuroPuzzle, Orientation
from task3_kakuro.csp import KakuroCSP, SumCombinationTable
from task3_kakuro.search import BacktrackingSolver
from task3_kakuro.verify import SolutionVerifier
from task3_kakuro.__main__ import format_solution_grid

puzzles_dir = Path('../src/task3_kakuro/data/puzzles')
if not puzzles_dir.exists():
    puzzles_dir = Path('src/task3_kakuro/data/puzzles')

small_puzzle = KakuroPuzzle.from_json(puzzles_dir / 'small_5x5.json')
large_puzzle = KakuroPuzzle.from_json(puzzles_dir / 'large_9x9.json')

for pz in (small_puzzle, large_puzzle):
    across_runs = [r for r in pz.runs() if r.orientation == Orientation.ACROSS]
    down_runs = [r for r in pz.runs() if r.orientation == Orientation.DOWN]
    print(f'Puzzle: {pz.name} ({pz.rows}x{pz.cols})')
    print(f'  - Variables (White cells): {len(pz.white_cells())}')
    print(f'  - Constraints (Runs): {len(pz.runs())} ({len(across_runs)} across, {len(down_runs)} down)')
    print(f'  - Total Across Clue Sum: {sum(r.clue for r in across_runs)} | Down Clue Sum: {sum(r.clue for r in down_runs)}')

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.7)
Puzzle: small_5x5 (5x5)
  - Variables (White cells): 12
  - Constraints (Runs): 8 (4 across, 4 down)
  - Total Across Clue Sum: 60 | Down Clue Sum: 60
Puzzle: large_9x9 (9x9)
  - Variables (White cells): 48
  - Constraints (Runs): 28 (14 across, 14 down)
  - Total Across Clue Sum: 249 | Down Clue Sum: 249


## 1. Solving and Verifying Both Official Kakuro Boards
We run the full `mac-mrvdeg-lcv` solver (Generalized Arc Consistency + MRV with Degree tie-breaking + Least Constraining Value) and validate the solution using the independent `SolutionVerifier`.

In [2]:
for pz in (small_puzzle, large_puzzle):
    csp = KakuroCSP(pz)
    solver = BacktrackingSolver(csp, preset='mac-mrvdeg-lcv', seed=0)
    solution = solver.solve()
    is_valid = SolutionVerifier.verify(pz, solution)
    st = solver.stats
    print(f'=== {pz.name.upper()} ===')
    print(f'Solved: {st.solved} | Independent Verification: {is_valid}')
    print(f'Nodes explored: {st.nodes} | Backtracks: {st.backtracks} | Time: {st.elapsed_s * 1000:.2f} ms')
    print(format_solution_grid(pz, solution))
    print()

=== SMALL_5X5 ===
Solved: True | Independent Verification: True
Nodes explored: 12 | Backtracks: 0 | Time: 1.45 ms
   #       #      21/     14/      #   
   #     14/12    [8]     [4]     11/  
  /21     [9]     [3]     [1]     [8]  
  /11     [5]     [1]     [2]     [3]  
   #      /16     [9]     [7]      #   

=== LARGE_9X9 ===
Solved: True | Independent Verification: True
Nodes explored: 48 | Backtracks: 0 | Time: 11.71 ms
   #       #      44/     16/     9/       #      15/     44/      #   
   #      /22     [5]     [9]     [8]    11/3     [1]     [2]     14/  
   #     17/28    [2]     [7]     [1]     [3]     [4]     [6]     [5]  
  /14     [8]     [6]      #     12/20    [1]     [2]     [8]     [9]  
  /17     [9]     [8]    29/22    [3]     [2]     [8]     [9]     7/   
   #     12/28    [9]     [8]     [6]     [5]     /12     [7]     [5]  
  /14     [3]     [4]     [5]     [2]     7/     12/6     [4]     [2]  
  /39     [9]     [3]     [7]     [1]     [6]     [8]     [5]   

## 2. Effectiveness & Stability Comparison Across 5 Solver Configurations
We compare five configurations across 30 random tie-breaking seeds (`seed = 0..29`):
- `bt`: Plain Backtracking Search (with combination-filtered initial domains and partial sum/alldiff consistency check).
- `fc`: Backtracking + Forward Checking (static variable order).
- `fc-mrv`: Backtracking + Forward Checking + Minimum Remaining Values (MRV).
- `mac-mrv`: Backtracking + Generalized Arc Consistency (MAC) + MRV.
- `mac-mrvdeg-lcv`: Backtracking + Generalized Arc Consistency (MAC) + MRV-Degree + Least Constraining Value (LCV).

In [3]:
csv_path = Path('results/task3/task3_kakuro_benchmark.csv')
if not csv_path.exists():
    csv_path = Path('experiments/results/task3/task3_kakuro_benchmark.csv')

with csv_path.open('r', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

print(f'Loaded {len(records)} benchmark records from {csv_path.name}\n')
header = f"{'Puzzle':<11} | {'Preset':<15} | {'Runs':>4} | {'Solved':>6} | {'Mean Nodes':>10} | {'Mean BT':>8} | {'Std Nodes':>9} | {'Mean ms':>8}"
print(header)
print('-' * len(header))

for pz_name in ('small_5x5', 'large_9x9'):
    for preset in ('bt', 'fc', 'fc-mrv', 'mac-mrv', 'mac-mrvdeg-lcv'):
        sub = [r for r in records if r['puzzle'] == pz_name and r['preset'] == preset]
        nodes = [int(r['nodes']) for r in sub]
        bts = [int(r['backtracks']) for r in sub]
        times = [float(r['elapsed_ms']) for r in sub]
        solved_rate = sum(1 for r in sub if r['verified'] == 'True') / len(sub) * 100.0
        mean_n = statistics.mean(nodes)
        mean_bt = statistics.mean(bts)
        std_n = statistics.pstdev(nodes) if len(nodes) > 1 else 0.0
        mean_t = statistics.mean(times)
        print(f'{pz_name:<11} | {preset:<15} | {len(sub):>4} | {solved_rate:>5.0f}% | {mean_n:>10.1f} | {mean_bt:>8.1f} | {std_n:>9.2f} | {mean_t:>8.2f}')
    print('-' * len(header))

Loaded 184 benchmark records from task3_kakuro_benchmark.csv

Puzzle      | Preset          | Runs | Solved | Mean Nodes |  Mean BT | Std Nodes |  Mean ms
--------------------------------------------------------------------------------------------
small_5x5   | bt              |    1 |   100% |     4692.0 |    921.0 |      0.00 |    32.71
small_5x5   | fc              |    1 |   100% |       22.0 |     10.0 |      0.00 |     1.55
small_5x5   | fc-mrv          |   30 |   100% |       12.1 |      0.1 |      0.30 |     0.88
small_5x5   | mac-mrv         |   30 |   100% |       12.0 |      0.0 |      0.00 |     1.16
small_5x5   | mac-mrvdeg-lcv  |   30 |   100% |       12.0 |      0.0 |      0.00 |     1.41
--------------------------------------------------------------------------------------------
large_9x9   | bt              |    1 |   100% |     2485.0 |    579.0 |      0.00 |    22.42
large_9x9   | fc              |    1 |   100% |       96.0 |     48.0 |      0.00 |    10.14
large_9x